# 06 - Practical Coding Exercises

This notebook contains practical implementations of Scikit-learn and TensorFlow concepts relevant to the Machine Learning Software Developer role.

## 1. Scikit-Learn Pipeline & Feature Engineering

**Task:** Create a pipeline that handles missing values, scales numerical features, and trains a Random Forest Classifier.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Create a dummy dataset
data = {
    'age': [25, 30, np.nan, 45, 50, 23, 35, np.nan, 40, 48],
    'salary': [50000, 60000, 70000, np.nan, 80000, 52000, 58000, 75000, 62000, np.nan],
    'city': ['NY', 'SF', 'NY', 'LA', 'SF', 'LA', 'NY', 'SF', 'LA', 'NY'],
    'purchased': [0, 1, 0, 1, 1, 0, 0, 1, 1, 0]
}
df = pd.DataFrame(data)

X = df.drop('purchased', axis=1)
y = df['purchased']

# Identify numeric and categorical columns
numeric_features = ['age', 'salary']
categorical_features = ['city']

# Define transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Final Pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

## 2. TensorFlow Deep Learning

**Task:** Build a simple neural network for binary classification using the Keras Sequential API with Dropout and Early Stopping.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Simulate input data
X_train_dl = np.random.random((1000, 10))
y_train_dl = np.random.randint(2, size=(1000, 1))

def build_model(input_shape):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_shape,)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_model(X_train_dl.shape[1])

# Callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train_dl, y_train_dl,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

print("Training completed.")
print(f"Best Val Accuracy: {max(history.history['val_accuracy']):.4f}")

## 3. Data Drift Detection (Conceptual Script)

**Task:** How would you detect if the distribution of a feature has changed?

In [ ]:
from scipy.stats import ks_2samp

def detect_drift(reference_data, current_data, threshold=0.05):
    """Uses Kolmogorov-Smirnov test to detect drift."""
    statistic, p_value = ks_2samp(reference_data, current_data)
    
    if p_value < threshold:
        return True, p_value # Drift detected
    return False, p_value

ref = np.random.normal(0, 1, 1000)
curr = np.random.normal(0.5, 1, 1000) # Shifted mean

is_drift, p = detect_drift(ref, curr)
print(f"Drift Detected: {is_drift} (p-value: {p:.4f})")